In [ ]:
from google.colab import files
print("Upload olist.db first:")
uploaded1 = files.upload()

Upload olist.db first:


Saving olist.db to olist.db


In [ ]:
print("Now upload chroma_db_reviews.zip:")
uploaded2 = files.upload()

Now upload chroma_db_reviews.zip:


Saving chroma_db_reviews.zip to chroma_db_reviews.zip


In [ ]:
!ls -la /content/

total 110192
drwxr-xr-x 1 root root      4096 Sep 20 10:54 .
drwxr-xr-x 1 root root      4096 Sep 20 10:25 ..
-rw-r--r-- 1 root root   1071911 Sep 20 10:54 chroma_db_reviews.zip
drwxr-xr-x 4 root root      4096 Sep 16 13:26 .config
-rw-r--r-- 1 root root 111747072 Sep 20 10:52 olist.db
drwxr-xr-x 1 root root      4096 Sep 16 13:26 sample_data


In [ ]:
import zipfile
with zipfile.ZipFile("chroma_db_reviews.zip", 'r') as zip_ref:
    zip_ref.extractall("./chroma_db_reviews")

!find ./chroma_db_reviews -type f

./chroma_db_reviews/chroma_db_reviews/b0ce061e-1667-4fd3-a3f6-92a2776cb3b5/link_lists.bin
./chroma_db_reviews/chroma_db_reviews/b0ce061e-1667-4fd3-a3f6-92a2776cb3b5/data_level0.bin
./chroma_db_reviews/chroma_db_reviews/b0ce061e-1667-4fd3-a3f6-92a2776cb3b5/header.bin
./chroma_db_reviews/chroma_db_reviews/b0ce061e-1667-4fd3-a3f6-92a2776cb3b5/length.bin
./chroma_db_reviews/chroma_db_reviews/chroma.sqlite3


In [ ]:
!pip install -q huggingface_hub sqlalchemy pandas chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [ ]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:////content/olist.db")

with engine.connect() as conn:
    result = conn.exec_driver_sql("SELECT COUNT(*) FROM orders;")
    print("Orders table check:", result.fetchone())

Orders table check: (99441,)


In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

from huggingface_hub import InferenceClient
client = InferenceClient(token=HF_TOKEN)
MODEL = "meta-llama/Llama-3.3-70B-Instruct"

In [ ]:
def get_schema(engine):
    with engine.connect() as conn:
        tables = conn.exec_driver_sql("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
        schema_str = ""
        for (table,) in tables:
            cols = conn.exec_driver_sql(f"PRAGMA table_info({table});").fetchall()
            col_list = ", ".join(f"{c[1]} ({c[2]})" for c in cols)
            schema_str += f"Table {table}: {col_list}\n"
        return schema_str

schema = get_schema(engine)
print(schema)

Table customers: customer_id (TEXT), customer_unique_id (TEXT), customer_zip_code_prefix (BIGINT), customer_city (TEXT), customer_state (TEXT)
Table geolocation: geolocation_zip_code_prefix (BIGINT), geolocation_lat (FLOAT), geolocation_lng (FLOAT), geolocation_city (TEXT), geolocation_state (TEXT)
Table orders: order_id (TEXT), customer_id (TEXT), order_status (TEXT), order_purchase_timestamp (TEXT), order_approved_at (TEXT), order_delivered_carrier_date (TEXT), order_delivered_customer_date (TEXT), order_estimated_delivery_date (TEXT)
Table order_items: order_id (TEXT), order_item_id (BIGINT), product_id (TEXT), seller_id (TEXT), shipping_limit_date (TEXT), price (FLOAT), freight_value (FLOAT)
Table order_payments: order_id (TEXT), payment_sequential (BIGINT), payment_type (TEXT), payment_installments (BIGINT), payment_value (FLOAT)
Table order_reviews: review_id (TEXT), order_id (TEXT), review_score (BIGINT), review_comment_title (TEXT), review_comment_message (TEXT), review_creatio

In [ ]:
SYSTEM_PROMPT = """You are a SQL expert working with the Olist e-commerce database.

Schema:
{schema}

Important notes:
- product_category_name is in Portuguese. To get English category names, join products.product_category_name to product_category_name_translation.product_category_name, and use product_category_name_english.
- order_delivered_customer_date vs order_estimated_delivery_date determines if a delivery was late. Only compare these when order_delivered_customer_date IS NOT NULL, since undelivered/cancelled orders have NULL dates and should be excluded from "late/on-time" comparisons, not counted as on-time.
- "Revenue" means product price only (oi.price), not including freight_value, unless the question explicitly asks about shipping/freight costs.
- When counting "number of orders", use COUNT(DISTINCT order_id), not COUNT(order_id) — an order can contain multiple items, which would otherwise inflate the count.
- Be careful when joining order_items to order-level data (like order_reviews or orders) for averages/counts: an order can have multiple items, which duplicates rows and skews AVG() or COUNT() results. When aggregating order-level values, first get distinct order_ids before joining, or aggregate at the order level before joining to item-level category data. See Example 2 below for the correct pattern.
- To count unique/distinct customers (people), use customer_unique_id, not customer_id. customer_id is generated per-order, so the same person gets a different customer_id on each order — COUNT(DISTINCT customer_id) does not give the real number of unique customers.
- Only output a single valid SQLite query. No explanation, no markdown, no backticks.

Example 1:
Question: What is total revenue by English product category?
SQL: SELECT t.product_category_name_english, SUM(oi.price) as revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN product_category_name_translation t ON p.product_category_name = t.product_category_name
GROUP BY t.product_category_name_english
ORDER BY revenue DESC;

Example 2 (correct handling of order_items joined to order-level data, avoiding row duplication):
Question: Which product categories have the worst average review score?
SQL: SELECT t.product_category_name_english, AVG(orv.review_score) as avg_score
FROM (SELECT DISTINCT order_id, product_id FROM order_items) oi
JOIN products p ON oi.product_id = p.product_id
JOIN product_category_name_translation t ON p.product_category_name = t.product_category_name
JOIN order_reviews orv ON oi.order_id = orv.order_id
GROUP BY t.product_category_name_english
ORDER BY avg_score ASC;

Now write SQL for this question:
{question}
"""

In [ ]:
def generate_sql(question, schema):
    prompt = SYSTEM_PROMPT.format(schema=schema, question=question)
    response = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL,
        max_tokens=300,
        temperature=0
    )
    return response.choices[0].message.content.strip()

def run_query(sql, engine):
    forbidden = ["DROP", "DELETE", "UPDATE", "INSERT", "ALTER"]
    if any(word in sql.upper() for word in forbidden):
        return "Blocked: only SELECT queries are allowed."
    with engine.connect() as conn:
        result = conn.exec_driver_sql(sql)
        return result.fetchall(), list(result.keys())

In [ ]:
%%writefile retriever.py
import chromadb
from sentence_transformers import SentenceTransformer

client = chromadb.PersistentClient(path="./chroma_db_reviews")
collection = client.get_or_create_collection("customer_reviews")

embed_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

def retrieve_reviews(query: str, top_k: int = 3):
    query_vector = embed_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_vector, n_results=top_k)
    matched_reviews = []
    if results and "documents" in results and results["documents"]:
        for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
            matched_reviews.append({
                "review": doc,
                "order_id": meta.get("order_id"),
                "review_score": meta.get("review_score"),
                "predicted_stars": meta.get("predicted_stars")
            })
    return matched_reviews

Writing retriever.py


In [ ]:
from retriever import retrieve_reviews

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
results_log = []

In [ ]:
# Test SQL pipeline
sql = generate_sql("How many total orders are there?", schema)
print("SQL:", sql)
rows, cols = run_query(sql, engine)
print("SQL result:", rows)

# Test semantic search pipeline
reviews = retrieve_reviews("delayed delivery and late shipping", top_k=3)
print("\nSemantic search result:")
for r in reviews:
    print(r)

SQL: SELECT COUNT(DISTINCT order_id) as total_orders
FROM orders
SQL result: [(99441,)]

Semantic search result:


In [ ]:
import chromadb
client_check = chromadb.PersistentClient(path="./chroma_db_reviews")
collection_check = client_check.get_or_create_collection("customer_reviews")
print("Number of items in collection:", collection_check.count())

Number of items in collection: 0


In [ ]:
!find ./chroma_db_reviews -type f

./chroma_db_reviews/chroma.sqlite3
./chroma_db_reviews/chroma_db_reviews/b0ce061e-1667-4fd3-a3f6-92a2776cb3b5/link_lists.bin
./chroma_db_reviews/chroma_db_reviews/b0ce061e-1667-4fd3-a3f6-92a2776cb3b5/data_level0.bin
./chroma_db_reviews/chroma_db_reviews/b0ce061e-1667-4fd3-a3f6-92a2776cb3b5/header.bin
./chroma_db_reviews/chroma_db_reviews/b0ce061e-1667-4fd3-a3f6-92a2776cb3b5/length.bin
./chroma_db_reviews/chroma_db_reviews/chroma.sqlite3


In [ ]:
client_check = chromadb.PersistentClient(path="./chroma_db_reviews")
print("Available collections:", client_check.list_collections())

Available collections: [Collection(name=customer_reviews)]


In [ ]:
import chromadb
client_check = chromadb.PersistentClient(path="./chroma_db_reviews/chroma_db_reviews")
collection_check = client_check.get_or_create_collection("customer_reviews")
print("Number of items in collection:", collection_check.count())

Number of items in collection: 500


In [ ]:
%%writefile retriever.py
import chromadb
from sentence_transformers import SentenceTransformer

client = chromadb.PersistentClient(path="./chroma_db_reviews/chroma_db_reviews")
collection = client.get_or_create_collection("customer_reviews")

embed_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

def retrieve_reviews(query: str, top_k: int = 3):
    query_vector = embed_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_vector, n_results=top_k)
    matched_reviews = []
    if results and "documents" in results and results["documents"]:
        for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
            matched_reviews.append({
                "review": doc,
                "order_id": meta.get("order_id"),
                "review_score": meta.get("review_score"),
                "predicted_stars": meta.get("predicted_stars")
            })
    return matched_reviews

Overwriting retriever.py


In [ ]:
import importlib
import retriever
importlib.reload(retriever)
from retriever import retrieve_reviews

results = retrieve_reviews("delayed delivery and late shipping", top_k=3)
for r in results:
    print(r)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'review': 'comprei um produto para ser entregue por essa loja\r\ne ainda não recebi\r\npois assim já se passou a data de entrega', 'order_id': '793d395a896de12a846ed4f345bce550', 'review_score': '2', 'predicted_stars': '1'}
{'review': 'Sempre compro pela Internet e a entrega ocorre antes do prazo combinado, que acredito ser o prazo máximo. No stark o prazo máximo já se esgotou e ainda não recebi o produto.', 'order_id': '68e55ca79d04a79f20d4bfc0146f4b66', 'review_score': '1', 'predicted_stars': '1'}
{'review': 'Recebi exatamente o que esperava. As demais encomendas de outros vendedores atrasaram, mas esta chegou no prazo.', 'order_id': '37e7875cdce5a9e5b3a692971f370151', 'review_score': '4', 'predicted_stars': '4'}


In [ ]:
results_log.append({
    "component": "semantic_search",
    "test_query": "delayed delivery and late shipping",
    "status": "correct - retrieved relevant multilingual reviews with sentiment metadata attached"
})

In [ ]:
def ask_question(question, engine, schema):
    # Get the SQL answer
    sql = generate_sql(question, schema)
    rows, cols = run_query(sql, engine)

    # Get supporting review evidence
    reviews = retrieve_reviews(question, top_k=3)

    return {
        "question": question,
        "sql": sql,
        "sql_result": rows[:5],
        "supporting_reviews": reviews
    }

In [ ]:
result = ask_question("Which product categories have the most late deliveries?", engine, schema)
print(result)

{'question': 'Which product categories have the most late deliveries?', 'sql': 'SELECT t.product_category_name_english, COUNT(DISTINCT oi.order_id) as late_deliveries\nFROM order_items oi\nJOIN products p ON oi.product_id = p.product_id\nJOIN product_category_name_translation t ON p.product_category_name = t.product_category_name\nJOIN orders o ON oi.order_id = o.order_id\nWHERE o.order_delivered_customer_date > o.order_estimated_delivery_date\nGROUP BY t.product_category_name_english\nORDER BY late_deliveries DESC;', 'sql_result': [('bed_bath_table', 811), ('health_beauty', 776), ('sports_leisure', 584), ('furniture_decor', 535), ('computers_accessories', 503)], 'supporting_reviews': [{'review': 'Otimo produto e entrega no prazo', 'order_id': '0a1bbcb7a1af6b58931664e147542a2d', 'review_score': '5', 'predicted_stars': '5'}, {'review': 'Produto entregue dentro do prazo .', 'order_id': 'c942d232f05adc081f767f145dd8fa36', 'review_score': '5', 'predicted_stars': '4'}, {'review': 'entrega r

In [ ]:
def ask_question(question, engine, schema, review_query=None, include_reviews=True, filter_negative=False):
    sql = generate_sql(question, schema)
    rows, cols = run_query(sql, engine)

    reviews = []
    if include_reviews:
        search_query = review_query if review_query else question
        reviews = retrieve_reviews(search_query, top_k=5)
        if filter_negative:
            reviews = [r for r in reviews if r.get("review_score") and int(r["review_score"]) <= 3]
        reviews = reviews[:3]

    return {
        "question": question,
        "sql": sql,
        "sql_result": rows[:5],
        "supporting_reviews": reviews
    }

In [ ]:
result = ask_question(
    "Which product categories have the most late deliveries?",
    engine, schema,
    review_query="late delivery complaint product never arrived delayed"
)
print(result)

{'question': 'Which product categories have the most late deliveries?', 'sql': 'SELECT t.product_category_name_english, COUNT(DISTINCT oi.order_id) as late_deliveries\nFROM order_items oi\nJOIN products p ON oi.product_id = p.product_id\nJOIN product_category_name_translation t ON p.product_category_name = t.product_category_name\nJOIN orders o ON oi.order_id = o.order_id\nWHERE o.order_delivered_customer_date > o.order_estimated_delivery_date\nGROUP BY t.product_category_name_english\nORDER BY late_deliveries DESC;', 'sql_result': [('bed_bath_table', 811), ('health_beauty', 776), ('sports_leisure', 584), ('furniture_decor', 535), ('computers_accessories', 503)], 'supporting_reviews': [{'review': 'meu produto veio com defeito,devolvi conforme informado.postado no correio conforme indicado.e até hoje não recebi nenhuma notificação da empresa quanto a devolução e análise para ter meuressarcimento', 'order_id': '475f773f6c0d909e49b758adb2401e3a', 'review_score': '1', 'predicted_stars': '1

In [ ]:
results_log.append({
    "question": "Which product categories have the most late deliveries?",
    "sql": sql,
    "sql_result": rows[:5],
    "supporting_reviews": result["supporting_reviews"],
    "status": "correct - combined SQL + review evidence, both align"
})

In [ ]:
results_log = []

In [ ]:
question = "How many total orders are there?"
result = ask_question(question, engine, schema, include_reviews=False)
print(result)

{'question': 'How many total orders are there?', 'sql': 'SELECT COUNT(DISTINCT order_id) as total_orders\nFROM orders', 'sql_result': [(99441,)], 'supporting_reviews': []}


In [ ]:
results_log.append({
    "question": question,
    "sql": result["sql"],
    "status": "correct",
    "top_result": result["sql_result"][0]
})

In [ ]:
question = "How many unique customers are there?"
result = ask_question(question, engine, schema, include_reviews=False)
print(result)

{'question': 'How many unique customers are there?', 'sql': 'SELECT COUNT(DISTINCT customer_unique_id) FROM customers', 'sql_result': [(96096,)], 'supporting_reviews': []}


In [ ]:
results_log.append({
    "question": "How many unique customers are there?",
    "sql": result["sql"],
    "status": "correct (after prompt fix)",
    "top_result": result["sql_result"][0]
})

In [ ]:
question = "What are the different order statuses and how many orders fall into each?"
result = ask_question(question, engine, schema, include_reviews=False)
print(result)

{'question': 'What are the different order statuses and how many orders fall into each?', 'sql': 'SELECT order_status, COUNT(DISTINCT order_id) as number_of_orders\nFROM orders\nGROUP BY order_status\nORDER BY number_of_orders DESC;', 'sql_result': [('delivered', 96478), ('shipped', 1107), ('canceled', 625), ('unavailable', 609), ('invoiced', 314)], 'supporting_reviews': []}


In [ ]:
results_log.append({
    "question": "What are the different order statuses and how many orders fall into each?",
    "sql": result["sql"],
    "status": "correct",
    "top_result": result["sql_result"][0]
})

In [ ]:
question = "What is the average delivery time by state?"
result = ask_question(question, engine, schema, include_reviews=False)
print(result)

{'question': 'What is the average delivery time by state?', 'sql': 'SELECT c.customer_state, AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)) as average_delivery_time\nFROM orders o\nJOIN customers c ON o.customer_id = c.customer_id\nWHERE o.order_delivered_customer_date IS NOT NULL\nGROUP BY c.customer_state\nORDER BY average_delivery_time ASC;', 'sql_result': [('SP', 8.761356563795475), ('PR', 11.991582227413579), ('MG', 12.010258342917206), ('DF', 12.967568108972717), ('SC', 14.959297453377435)], 'supporting_reviews': []}


In [ ]:
results_log.append({
    "question": "What is the average delivery time by state?",
    "sql": result["sql"],
    "status": "correct",
    "top_result": result["sql_result"][0]
})

In [ ]:
question = "What is total revenue by English product category?"
result = ask_question(question, engine, schema, include_reviews=False)
print(result)

{'question': 'What is total revenue by English product category?', 'sql': 'SELECT t.product_category_name_english, SUM(oi.price) as revenue\nFROM order_items oi\nJOIN products p ON oi.product_id = p.product_id\nJOIN product_category_name_translation t ON p.product_category_name = t.product_category_name\nGROUP BY t.product_category_name_english\nORDER BY revenue DESC;', 'sql_result': [('health_beauty', 1258681.34), ('watches_gifts', 1205005.68), ('bed_bath_table', 1036988.68), ('sports_leisure', 988048.97), ('computers_accessories', 911954.32)], 'supporting_reviews': []}


In [ ]:
results_log.append({
    "question": "What is total revenue by English product category?",
    "sql": result["sql"],
    "status": "correct",
    "top_result": result["sql_result"][0]
})

In [ ]:
question = "Which payment type is used most often?"
result = ask_question(question, engine, schema, include_reviews=False)
print(result)

{'question': 'Which payment type is used most often?', 'sql': 'SELECT payment_type, COUNT(DISTINCT order_id) as frequency\nFROM order_payments\nGROUP BY payment_type\nORDER BY frequency DESC\nLIMIT 1;', 'sql_result': [('credit_card', 76505)], 'supporting_reviews': []}


In [ ]:
results_log.append({
    "question": "Which payment type is used most often?",
    "sql": result["sql"],
    "status": "correct (more accurate than earlier run, due to COUNT DISTINCT fix)",
    "top_result": result["sql_result"][0]
})

In [ ]:
question = "What is the average review score across all orders?"
result = ask_question(question, engine, schema, include_reviews=False)
print(result)

{'question': 'What is the average review score across all orders?', 'sql': 'SELECT AVG(review_score) as average_review_score\nFROM order_reviews', 'sql_result': [(4.08642062404257,)], 'supporting_reviews': []}


In [ ]:
results_log.append({
    "question": "What is the average review score across all orders?",
    "sql": result["sql"],
    "status": "correct",
    "top_result": result["sql_result"][0]
})

In [ ]:
question = "Which sellers have the highest number of orders?"
result = ask_question(question, engine, schema, include_reviews=False)
print(result)

{'question': 'Which sellers have the highest number of orders?', 'sql': 'SELECT s.seller_id, COUNT(DISTINCT oi.order_id) as num_orders\nFROM order_items oi\nJOIN sellers s ON oi.seller_id = s.seller_id\nGROUP BY s.seller_id\nORDER BY num_orders DESC;', 'sql_result': [('6560211a19b47992c3666cc44a7e94c0', 1854), ('4a3ca9315b744ce9f8e9374361493884', 1806), ('cc419e0650a3c5ba77189a1882b7556a', 1706), ('1f50f920176fa81dab994f9023523100', 1404), ('da8622b14eb17ae2831f4ac5b9dab84a', 1314)], 'supporting_reviews': []}


In [ ]:
results_log.append({
    "question": "Which sellers have the highest number of orders?",
    "sql": result["sql"],
    "status": "correct",
    "top_result": result["sql_result"][0]
})

In [ ]:
question = "What is the average freight value by product category?"
result = ask_question(question, engine, schema, include_reviews=False)
print(result)

{'question': 'What is the average freight value by product category?', 'sql': 'SELECT t.product_category_name_english, AVG(oi.freight_value) as average_freight_value\nFROM order_items oi\nJOIN products p ON oi.product_id = p.product_id\nJOIN product_category_name_translation t ON p.product_category_name = t.product_category_name\nGROUP BY t.product_category_name_english\nORDER BY average_freight_value DESC;', 'sql_result': [('computers', 48.45467980295566), ('home_appliances_2', 44.53857142857143), ('furniture_mattress_and_upholstery', 42.90684210526316), ('kitchen_dining_laundry_garden_furniture', 42.702597864768684), ('furniture_bedroom', 42.49752293577981)], 'supporting_reviews': []}


In [ ]:
results_log.append({
    "question": "What is the average freight value by product category?",
    "sql": result["sql"],
    "status": "correct",
    "top_result": result["sql_result"][0]
})

In [ ]:
question = "Which product categories have the most late deliveries?"
result = ask_question(question, engine, schema, review_query="late delivery complaint product never arrived delayed", filter_negative=True)
print(result)

{'question': 'Which product categories have the most late deliveries?', 'sql': 'SELECT t.product_category_name_english, COUNT(DISTINCT oi.order_id) as late_deliveries\nFROM order_items oi\nJOIN products p ON oi.product_id = p.product_id\nJOIN product_category_name_translation t ON p.product_category_name = t.product_category_name\nJOIN orders o ON oi.order_id = o.order_id\nWHERE o.order_delivered_customer_date IS NOT NULL AND o.order_delivered_customer_date > o.order_estimated_delivery_date\nGROUP BY t.product_category_name_english\nORDER BY late_deliveries DESC;', 'sql_result': [('bed_bath_table', 811), ('health_beauty', 776), ('sports_leisure', 584), ('furniture_decor', 535), ('computers_accessories', 503)], 'supporting_reviews': [{'review': 'meu produto veio com defeito,devolvi conforme informado.postado no correio conforme indicado.e até hoje não recebi nenhuma notificação da empresa quanto a devolução e análise para ter meuressarcimento', 'order_id': '475f773f6c0d909e49b758adb2401

In [ ]:
results_log.append({
    "question": "Which product categories have the most late deliveries?",
    "sql": result["sql"],
    "sql_result": result["sql_result"][0],
    "supporting_reviews": result["supporting_reviews"],
    "status": "correct - combined SQL + review evidence, standout example"
})

In [ ]:
question = "Which product categories have the worst average review score?"
result = ask_question(question, engine, schema, review_query="bad product complaint disappointed poor quality", filter_negative=True)
print(result)

{'question': 'Which product categories have the worst average review score?', 'sql': 'SELECT t.product_category_name_english, AVG(orv.review_score) as avg_score\nFROM (SELECT DISTINCT order_id, product_id FROM order_items) oi\nJOIN products p ON oi.product_id = p.product_id\nJOIN product_category_name_translation t ON p.product_category_name = t.product_category_name\nJOIN order_reviews orv ON oi.order_id = orv.order_id\nGROUP BY t.product_category_name_english\nORDER BY avg_score ASC;', 'sql_result': [('security_and_services', 2.5), ('office_furniture', 3.595531587057011), ('fashion_male_clothing', 3.639344262295082), ('fashio_female_clothing', 3.7142857142857144), ('diapers_and_hygiene', 3.740740740740741)], 'supporting_reviews': [{'review': 'O produto não é bom.', 'order_id': 'f2e503ec5863ccf71c41fbe188c1879f', 'review_score': '3', 'predicted_stars': '2'}, {'review': 'O produto é de muita baixa qualidade', 'order_id': '3b715ecd6add927d4a2634e826487231', 'review_score': '2', 'predict

In [ ]:
results_log.append({
    "question": "Which product categories have the worst average review score?",
    "sql": result["sql"],
    "sql_result": result["sql_result"][0],
    "supporting_reviews": result["supporting_reviews"],
    "status": "correct (after adding worked example to prompt)"
})

In [ ]:
question = "Is there a relationship between delivery delay and review score?"
result = ask_question(question, engine, schema, review_query="late delivery complaint disappointed unhappy", filter_negative=True)
print(result)

{'question': 'Is there a relationship between delivery delay and review score?', 'sql': "SELECT \n  CASE \n    WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 'Late'\n    WHEN o.order_delivered_customer_date IS NOT NULL AND o.order_delivered_customer_date <= o.order_estimated_delivery_date THEN 'On Time'\n    ELSE 'Undelivered/Cancelled'\n  END AS delivery_status,\n  AVG(orv.review_score) AS avg_review_score\nFROM \n  orders o\n  JOIN order_reviews orv ON o.order_id = orv.order_id\nGROUP BY \n  CASE \n    WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 'Late'\n    WHEN o.order_delivered_customer_date IS NOT NULL AND o.order_delivered_customer_date <= o.order_estimated_delivery_date THEN 'On Time'\n    ELSE 'Undelivered/Cancelled'\n  END\nORDER BY \n  avg_review_score DESC;", 'sql_result': [('On Time', 4.293577567732184), ('Late', 2.566549798727438), ('Undelivered/Cancelled', 1.7612565445026178)], 'supporting_reviews': [{'review': 

In [ ]:
results_log.append({
    "question": "Is there a relationship between delivery delay and review score?",
    "sql": result["sql"],
    "sql_result": result["sql_result"],
    "supporting_reviews": result["supporting_reviews"],
    "status": "correct - strong standout example, added Undelivered/Cancelled as third category"
})

In [ ]:
question = "Which state has the highest total revenue?"
result = ask_question(question, engine, schema, include_reviews=False)
print(result)

{'question': 'Which state has the highest total revenue?', 'sql': 'SELECT c.customer_state, SUM(oi.price) as total_revenue\nFROM order_items oi\nJOIN orders o ON oi.order_id = o.order_id\nJOIN customers c ON o.customer_id = c.customer_id\nGROUP BY c.customer_state\nORDER BY total_revenue DESC\nLIMIT 1', 'sql_result': [('SP', 5202955.05)], 'supporting_reviews': []}


In [ ]:
results_log.append({
    "question": "Which state has the highest total revenue?",
    "sql": result["sql"],
    "status": "correct",
    "top_result": result["sql_result"][0]
})

In [ ]:
question = "What percentage of orders are delivered late overall?"
result = ask_question(question, engine, schema, include_reviews=False)
print(result)

{'question': 'What percentage of orders are delivered late overall?', 'sql': 'SELECT CAST(SUM(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END) AS REAL) * 100 / COUNT(o.order_id) AS late_delivery_percentage\nFROM orders o\nWHERE o.order_delivered_customer_date IS NOT NULL', 'sql_result': [(8.112898544715785,)], 'supporting_reviews': []}


In [ ]:
results_log.append({
    "question": "What percentage of orders are delivered late overall?",
    "sql": result["sql"],
    "status": "correct (percentage is of delivered orders, not all orders — accurate framing)",
    "top_result": result["sql_result"][0]
})

In [ ]:
question = "Which product category has the highest average order value, and what's its average review score?"
result = ask_question(question, engine, schema, include_reviews=False)
print(result)

{'question': "Which product category has the highest average order value, and what's its average review score?", 'sql': 'SELECT t.product_category_name_english, AVG(oi.price) as avg_order_value, AVG(orv.review_score) as avg_review_score\nFROM (SELECT DISTINCT oi.order_id, oi.product_id, oi.price \n      FROM order_items oi \n      JOIN orders o ON oi.order_id = o.order_id \n      WHERE o.order_delivered_customer_date IS NOT NULL) oi\nJOIN products p ON oi.product_id = p.product_id\nJOIN product_category_name_translation t ON p.product_category_name = t.product_category_name\nJOIN order_reviews orv ON oi.order_id = orv.order_id\nGROUP BY t.product_category_name_english\nORDER BY avg_order_value DESC\nLIMIT 1;', 'sql_result': [('computers', 1112.4148850574713, 4.224137931034483)], 'supporting_reviews': []}


In [ ]:
results_log.append({
    "question": "Which product category has the highest average order value, and what's its average review score?",
    "sql": result["sql"],
    "status": "correct - correctly combined DISTINCT subquery and NULL filtering patterns without explicit prompting for this exact case",
    "top_result": result["sql_result"][0]
})

In [ ]:
import sqlite3
import shutil
import os

# work on a copy so your original olist.db stays untouched
shutil.copy("olist.db", "olist_app.db")

conn = sqlite3.connect("olist_app.db")
conn.execute("DROP TABLE IF EXISTS geolocation;")
conn.execute("VACUUM;")
conn.close()

size_mb = os.path.getsize("olist_app.db") / (1024 * 1024)
print(f"New size: {size_mb:.1f} MB")

New size: 65.2 MB


In [ ]:
import os
os.rename("olist_app.db", "olist.db")